# Optical Scattering Sensor Simulation

An interactive walk-through of all six pipeline layers.

**Run cells in order.** Each cell demonstrates one layer and shows
a matplotlib plot of the result. Tweak parameters and re-run to explore.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from optical_metrology.illumination import Laser, GaussianBeamProfile
from optical_metrology.surface import RoughSurface, Material
from optical_metrology.scattering import LambertianScattering
from optical_metrology.optics import OpticalSystem, GaussianPSF, OpticalPropagator
from optical_metrology.detector import CMOSDetector
from optical_metrology.analysis import ImageAnalyzer, HistogramAnalyzer

np.random.seed(42)
print("All imports OK")

---
## Layer 1 — Illumination

Create a green laser and generate its light field on a 64×64 grid.

In [ ]:
laser = Laser(
    wavelength=532e-9,
    power=5e-3,
    beam_profile=GaussianBeamProfile(w0=3.0),
)
laser.propagation_direction = np.array([0.0, 0.0, -1.0])
lf = laser.generate_light_field(shape=(64, 64), spacing=0.4)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(lf.intensity, cmap='inferno', aspect='equal')
ax.set_title('Laser intensity  (532 nm, 5 mW)')
plt.colorbar(im, ax=ax, label='W / m²')
fig.tight_layout()

---
## Layer 2 — Surface geometry

Model a rough silicon surface.

In [ ]:
surface = RoughSurface(
    (64, 64), sigma=6.0, amplitude=0.5,
    material=Material('silicon'),
)

fig, ax = plt.subplots(figsize=(5, 4.5))
vlim = max(abs(surface.height.min()), abs(surface.height.max()))
im = ax.imshow(surface.height, cmap='RdBu_r',
               norm=Normalize(-vlim, vlim), aspect='equal')
ax.set_title(f'Surface height  (roughness = {surface.roughness:.4f})')
plt.colorbar(im, ax=ax, label='µm')
fig.tight_layout()

---
## Layer 3 — Scattering

Evaluate Lambertian scattering toward the observer.

In [ ]:
model = LambertianScattering(albedo=0.7)
scattered = model.evaluate(
    lf, surface,
    view_direction=np.array([0.0, 0.0, 1.0]),
)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(scattered.radiance, cmap='plasma', aspect='equal')
ax.set_title('Scattered radiance  (Lambertian, albedo=0.7)')
plt.colorbar(im, ax=ax, label='W / sr / m²')
fig.tight_layout()

---
## Layer 4 — Optics

Propagate through an optical system with a Gaussian PSF.

In [ ]:
optics = OpticalSystem(
    focal_length=0.05, aperture_diameter=0.008, wavelength=532e-9,
)
propagator = OpticalPropagator(psf_model=GaussianPSF(sigma=1.5))
sensor = propagator.propagate(scattered, optics)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(sensor.irradiance, cmap='viridis', aspect='equal')
ax.set_title('Sensor-plane irradiance')
plt.colorbar(im, ax=ax, label='W / m²')
fig.tight_layout()

---
## Layer 5 — Detector

Convert the optical field into a digital image.

The pipeline below is printed for reference.

In [ ]:
detector = CMOSDetector(
    exposure_time=2e-5,
    quantum_efficiency=0.9,
    dark_current=5.0,
    read_noise_sigma=2.0,
    full_well_capacity=80000.0,
    gain=1.0,
    bit_depth=12,
)

print(detector.pipeline_describe())
print()

image = detector.capture(sensor)
print(f'Output:  {image.pixels.shape}  {image.pixels.dtype}')
print(f'Range:   {image.pixels.min()} – {image.pixels.max()} ADU')

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(image.pixels, cmap='gray', aspect='equal')
ax.set_title(f'Digital image  ({detector.bit_depth}-bit)')
plt.colorbar(im, ax=ax, label='ADU')
fig.tight_layout()

---
## Layer 6 — Analysis

Compute pixel-value statistics and a histogram.

In [ ]:
analyzer = ImageAnalyzer(modules=[HistogramAnalyzer()])
report = analyzer.analyze(image)
hist = report.histogram
values = np.unique(image.pixels)

print('Measurements:')
for key, val in sorted(report.measurements.items()):
    label = key.replace('_', ' ').title()
    print(f'  {label:<20} {val:>10.4g}')
print(f'  Histogram bins     {len(hist)}')

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.bar(values, hist, width=1.0, color='steelblue', edgecolor='none')
ax.set_xlabel('Pixel value (ADU)')
ax.set_ylabel('Count')
ax.set_title('Pixel-value histogram')
fig.tight_layout()

---
## All layers combined

A single 2×3 figure summarising every stage of the pipeline.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle('Optical Scattering Sensor — Pipeline', fontsize=14, y=0.98)

plots = [
    (axes[0, 0], lf.intensity, '1  Illumination', 'inferno', 'W / m²', None),
    (axes[0, 1], surface.height, '2  Surface height', 'RdBu_r', 'µm',
     Normalize(-vlim, vlim)),
    (axes[0, 2], scattered.radiance, '3  Scattered radiance', 'plasma',
     'W / sr / m²', None),
    (axes[1, 0], sensor.irradiance, '4  Sensor irradiance', 'viridis',
     'W / m²', None),
    (axes[1, 1], image.pixels.astype(float), '5  Digital image', 'gray',
     'ADU', None),
]

for ax, data, title, cmap, label, norm in plots:
    im = ax.imshow(data, cmap=cmap, aspect='equal', norm=norm)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label=label)

ax = axes[1, 2]
ax.bar(values, hist, width=1.0, color='steelblue', edgecolor='none')
ax.set_title('6  Histogram')
ax.set_xlabel('ADU'); ax.set_ylabel('Count')

plt.tight_layout(rect=[0, 0, 1, 0.95])

---
## Next steps

- Change the laser wavelength / power and re-run cells 1, 3–6.
- Adjust `sigma` and `amplitude` on the rough surface (cell 2).
- Try `FlatSurface` or `ParticleSurface` instead.
- Tweak `exposure_time` or `gain` on the detector (cell 5).
- Use the terminal playground for quick iteration:
  `python3 playground.py --demo`